In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import os

df = pd.read_csv('../data/my_iot_dataset.csv')

# Drop original label column, keep category
df = df.drop(columns=['label'])

# Check for any infinite values and replace
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(df.median(numeric_only=True))

print("Shape after cleaning:", df.shape)
print("Any nulls remaining:", df.isnull().sum().sum())

Shape after cleaning: (50000, 47)
Any nulls remaining: 0


In [2]:
# Feature columns (all except category)
FEATURE_COLS = [c for c in df.columns if c != 'category']

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['category'].values

print("Features shape:", X.shape)
print("Labels shape:  ", y.shape)
print("Feature columns:", FEATURE_COLS)

Features shape: (50000, 46)
Labels shape:   (50000,)
Feature columns: ['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight']


In [3]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Label encoding:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls} → {i}")

Label encoding:
  Benign → 0
  DDoS → 1
  DoS → 2
  Mirai → 3
  Recon → 4


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded   # keeps category balance
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

# Verify balance
unique, counts = np.unique(y_train, return_counts=True)
print("\nTrain class distribution:")
for cls, cnt in zip(le.classes_, counts):
    print(f"  {cls}: {cnt}")

X_train: (40000, 46)
X_test:  (10000, 46)

Train class distribution:
  Benign: 8000
  DDoS: 8000
  DoS: 8000
  Mirai: 8000
  Recon: 8000


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done!")
print(f"Train mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"Train std  (should be ~1): {X_train_scaled.std():.4f}")

Scaling done!
Train mean (should be ~0): 0.0000
Train std  (should be ~1): 0.9208


In [6]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save processed data
np.save('../data/processed/X_train.npy', X_train_scaled)
np.save('../data/processed/X_test.npy',  X_test_scaled)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy',  y_test)

# Save scaler and encoder for later use in agents
pickle.dump(scaler, open('../models/scaler.pkl', 'wb'))
pickle.dump(le,     open('../models/label_encoder.pkl', 'wb'))

# Save feature column names
pickle.dump(FEATURE_COLS, open('../models/feature_cols.pkl', 'wb'))

print("Saved:")
print("  ✓ data/processed/X_train.npy")
print("  ✓ data/processed/X_test.npy")
print("  ✓ data/processed/y_train.npy")
print("  ✓ data/processed/y_test.npy")
print("  ✓ models/scaler.pkl")
print("  ✓ models/label_encoder.pkl")
print("  ✓ models/feature_cols.pkl")

Saved:
  ✓ data/processed/X_train.npy
  ✓ data/processed/X_test.npy
  ✓ data/processed/y_train.npy
  ✓ data/processed/y_test.npy
  ✓ models/scaler.pkl
  ✓ models/label_encoder.pkl
  ✓ models/feature_cols.pkl
